In [ ]:
from google import genai
from google.genai import types
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd
import os
import json
import time
import base64
import sys
import importlib
sys.path.append('..')
import common.prompts as prompts
import common.file_processing as file_processing
import common.file_processing as utils
import common.ai as ai
importlib.reload(prompts)
importlib.reload(file_processing)
importlib.reload(utils)
importlib.reload(ai)

In [ ]:
load_dotenv(override=True)

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")

gemini_client = genai.Client()

gemini_model = os.getenv("GEMINI_MODEL")

images_folder_path = os.getenv("IMG_PATH")
output_folder_path = os.getenv("BASIC_OUTPUT_FOLDER_PATH")

In [ ]:
def gemini_nutritionist(image):
    try:

        response = ai.gemini_nutritionist(
            client=gemini_client,
            model=gemini_model,
            tokens=1024,
            temperature=0.1,
            prompt=prompts.UNIFIED_NUTRITION_PROMPT,
            image_byte=image
        )
        
        try:
            data = file_processing.parse_json(response)
        except Exception as e:
            return {'success': False, 'error': f"No valid JSON found in the response: {str(e)}"}

        return {
            'success': True,
            'description': data.get('description'),
            'calories': float(data.get('calories', 0)),
            'proteins': float(data.get('proteins', 0)),
            'carbohydrates': float(data.get('carbohydrates', 0)),
            'fats': float(data.get('fats', 0)),
            'serving_size': float(data.get('serving_size', 0))
        }
    except Exception as e:
        return {'success': False, 'error': str(e)}

In [ ]:
def analyze_image(image_path, index):
    """Complete chained analysis for one image"""
    start_time = time.time()
    file_name = os.path.basename(image_path)

    with open(image_path, "rb") as f:
            image_byte = f.read()

    print(f"\n🔄 Processing {index}: {file_name}")

    print("  Gemini Nutritionist...")
    response = gemini_nutritionist(image_byte)

    if not response['success']:
        return {'success': False, 'error': f"Gemini failed: {response['error']}", 'index': index}

    print(f"    → {response['success']}")
    if not response.get("description"):
        return {'success': False, 'error': "Gemini returned empty description", 'index': index}

    total_time = time.time() - start_time
    time.sleep(2)
    print(f"  ✅ Complete! {response['description']} in {total_time:.1f}s")

    return {
        'success': True,
        'id': index,
        'file_name': file_name,
        'response': response,
        'processing_time': total_time
    }

In [ ]:
def process_dataset(file_name, folder_path, start=1, end=None):
    """Process images with chained analysis"""

    total_images = file_processing.count_images(folder_path)
    if end is None:
        end = total_images
    end = min(end, total_images)
    
    print(f"🚀 Processing images {start} to {end} ({end-start+1} total)")
    
    results = []
    successful = 0
    
    for i in range(start, end + 1):
        image_path = os.path.join(folder_path, f"{i}.jpg")
        result = analyze_image(image_path, i)
        results.append(result)
        
        if result['success']:
            successful += 1
    
    print(f"\n🎉 Completed! {successful}/{len(results)} successful")
    
    # Save results
    output_file = f"{file_name}.json"
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    
    print(f"📁 Results saved to: {output_file}")
    return results

In [ ]:
def export_to_excel(file_name):
    """Export results to Excel"""
    with open(f"{file_name}.json", 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Prepare final results
    final_results = []
    for item in data:
        if item['success']:
            nutrition = item['response']
            final_results.append({
                'id': item['id'],
                'description': nutrition['description'],
                'serving_size': nutrition['serving_size'],
                'calories': nutrition['calories'],
                'proteins': nutrition['proteins'],
                'carbohydrates': nutrition['carbohydrates'],
                'fats': nutrition['fats']
            })
    
    # Create DataFrame and export
    df = pd.DataFrame(final_results)
    output_path = os.path.join(output_folder_path, f"{file_name}.xlsx")
    df.to_excel(output_path, index=False)
    
    print(f"✅ Excel exported: {output_path}")
    print(f"📊 {len(final_results)} successful analyses")
    
    return output_path

In [ ]:
file_name = "nutria_gemini"

In [ ]:
results = process_dataset(file_name, images_folder_path, start=1, end=file_processing.count_images(images_folder_path))
# results = process_dataset(file_name, images_folder_path, start=1, end=2)

In [ ]:
excel_path = export_to_excel(file_name)
print(f"\n🎯 Done! Check: {excel_path}")